# DataLoader Debug Notebook

`src/data/loader.py` 디버깅 노트북입니다.
이번 버전은 **개선된 k-fold(`year`/`year_half`) 경로를 기준**으로 점검합니다.

## 디버깅 체크리스트
1. 입력 데이터/컬럼 유효성 확인
2. feature/target 선택 결과 확인
3. 윈도우 생성(`_build_windows`) 결과 확인
4. term 생성(`_add_term_column`)과 fold 분할(`_fold_indices`) 확인
5. `DataModule.setup()`의 k-fold 결과(train/val/test) 확인
6. 배치 shape, 기본 assert, 실패 케이스 확인

In [1]:
from pathlib import Path
import sys
import os
from types import SimpleNamespace

import numpy as np
import pandas as pd

# project_root = Path('/data2/wk/SW_framework/workdir/sw-framework-v001')
# os.chdir(project_root)  # traceback 링크/상대경로 기준 통일


from src.data.loader import (
    DataModule,
    _build_windows,
    _add_term_column,
    _fold_indices,
)

import sys, os, inspect
import src.data.loader as loader


# print(sys.executable)
# print(os.environ.get("PYTHONPATH"))

In [2]:
# 디버그용 설정 (개선된 k-fold 기준)
cfg = SimpleNamespace(
    data_path='/data2/wk/SW_framework/data/xray_hourly.csv',
    target_col='Long',
    feature_cols=['Year', 'Month', 'Day', 'Hour'],
    time_col='Datetime',
    seq_len=24 * 7,
    pred_len=24 * 3,
    split_type='year_half',
    n_fold=5,
    fold_numb=0,
    train_ratio=0.7,   # split_type='ratio'일 때만 사용
    val_ratio=0.15,    # split_type='ratio'일 때만 사용
    batch_size=32,
    shuffle_train=True,
    num_workers=0,
)

print(cfg)

namespace(data_path='/data2/wk/SW_framework/data/xray_hourly.csv', target_col='Long', feature_cols=['Year', 'Month', 'Day', 'Hour'], time_col='Datetime', seq_len=168, pred_len=72, split_type='year_half', n_fold=5, fold_numb=0, train_ratio=0.7, val_ratio=0.15, batch_size=32, shuffle_train=True, num_workers=0)


In [3]:
# 1) 원본 CSV 로드/기본 검증
raw_df = pd.read_csv(cfg.data_path)
print('shape:', raw_df.shape)
print('columns:', raw_df.columns.tolist())
print(raw_df.head(3))

assert cfg.target_col in raw_df.columns, f"target_col '{cfg.target_col}' not found"

raw_df.loc[(0 > raw_df.Long.abs()) | (raw_df.Long.abs() > 2), 'Long'] = np.nan

shape: (184162, 7)
columns: ['Year', 'Month', 'Day', 'Hour', 'Long', 'Flare_Class', 'R_Level']
   Year  Month  Day  Hour          Long Flare_Class R_Level
0  2002     12    3     0  5.720000e-07         NaN     NaN
1  2002     12    3     1  6.730000e-07         NaN     NaN
2  2002     12    3     2  5.430000e-07         NaN     NaN


In [4]:
# 결측/이상치 상태 확인
nan_count = int(raw_df[cfg.target_col].isna().sum())
finite_mask = np.isfinite(raw_df[cfg.target_col].to_numpy(dtype=np.float64))
print('target nan count:', nan_count)
print('target finite ratio:', float(finite_mask.mean()))

# 디버깅용: 타깃의 기본 통계
print(raw_df[cfg.target_col].describe())

target nan count: 302
target finite ratio: 0.9983601394424474
count    1.838600e+05
mean     1.611014e-06
std      1.426292e-05
min      0.000000e+00
25%      5.600000e-08
50%      2.550000e-07
75%      9.750497e-07
max      1.840000e-03
Name: Long, dtype: float64


In [5]:
# 2) feature/target 선택 로직 디버깅
if cfg.feature_cols is None:
    exclude = {cfg.target_col}
    if cfg.time_col:
        exclude.add(cfg.time_col)
    feature_cols = [c for c in raw_df.columns if c not in exclude]
else:
    feature_cols = cfg.feature_cols

use_cols = feature_cols + [cfg.target_col]
print('feature_cols:', feature_cols)
print('use_cols:', use_cols)

for c in use_cols:
    assert c in raw_df.columns, f"missing column: {c}"

values = raw_df[use_cols].to_numpy(dtype=np.float32)
print('values shape:', values.shape)
print('nan count:', int(np.isnan(values).sum()))

feature_cols: ['Year', 'Month', 'Day', 'Hour']
use_cols: ['Year', 'Month', 'Day', 'Hour', 'Long']
values shape: (184162, 5)
nan count: 302


In [6]:
# 3) _build_windows 단계 디버깅
x_win, y_win = _build_windows(values, cfg.seq_len, cfg.pred_len)
print('x_win shape:', x_win.shape)  # (N, seq_len, D)
print('y_win shape:', y_win.shape)  # (N, pred_len, D)

# loader.py와 동일하게 target만 남기기
y_target = y_win[:, :, -1:]
print('y_target shape:', y_target.shape)

# # 샘플 인덱스 추적
# sample_i = 0
# print('\n[trace sample index=0]')
# print('x first row:', x_win[sample_i, 0, :])
# print('x last row :', x_win[sample_i, -1, :])
# print('y target   :', y_target[sample_i, :, 0])

x_win shape: (183923, 168, 5)
y_win shape: (183923, 72, 5)
y_target shape: (183923, 72, 1)


In [7]:
# 개선된 k-fold 로직용 term/fold 준비
raw_df_with_time = raw_df.copy()
raw_df_with_time['Datetime'] = pd.to_datetime(raw_df_with_time[['Year', 'Month', 'Day', 'Hour']])

term_df = _add_term_column(raw_df_with_time, time_col='Datetime', split_type=cfg.split_type)
terms = sorted(term_df['_term'].dropna().unique().tolist())
fold_map = _fold_indices(len(terms), n_fold=cfg.n_fold, fold_numb=cfg.fold_numb)

split_terms = {k: [terms[i] for i in idx.tolist()] for k, idx in fold_map.items()}
print('split_type:', cfg.split_type)
print('n_term:', len(terms), 'n_fold:', cfg.n_fold, 'fold_numb:', cfg.fold_numb)
print('train terms:', split_terms['train'])
print('val terms  :', split_terms['val'])
print('test terms :', split_terms['test'])

split_type: year_half
n_term: 46 n_fold: 5 fold_numb: 0
train terms: ['2002-H2', '2003-H1', '2003-H2', '2005-H1', '2005-H2', '2006-H1', '2007-H2', '2008-H1', '2008-H2', '2010-H1', '2010-H2', '2011-H1', '2012-H2', '2013-H1', '2013-H2', '2015-H1', '2015-H2', '2016-H1', '2017-H2', '2018-H1', '2018-H2', '2020-H1', '2020-H2', '2021-H1', '2022-H2', '2023-H1', '2023-H2', '2025-H1']
val terms  : ['2004-H1', '2006-H2', '2009-H1', '2011-H2', '2014-H1', '2016-H2', '2019-H1', '2021-H2', '2024-H1']
test terms : ['2004-H2', '2007-H1', '2009-H2', '2012-H1', '2014-H2', '2017-H1', '2019-H2', '2022-H1', '2024-H2']


In [8]:
# fold 회전 결과를 전체 fold에 대해 확인
for fn in range(cfg.n_fold):
    fm = _fold_indices(len(terms), n_fold=cfg.n_fold, fold_numb=fn)
    tr_terms = [terms[i] for i in fm['train'].tolist()]
    vl_terms = [terms[i] for i in fm['val'].tolist()]
    ts_terms = [terms[i] for i in fm['test'].tolist()]
    print(f"fold={fn:02d} | train={len(tr_terms)} val={len(vl_terms)} test={len(ts_terms)}")
    print('  val :', vl_terms)
    print('  test:', ts_terms)

fold=00 | train=28 val=9 test=9
  val : ['2004-H1', '2006-H2', '2009-H1', '2011-H2', '2014-H1', '2016-H2', '2019-H1', '2021-H2', '2024-H1']
  test: ['2004-H2', '2007-H1', '2009-H2', '2012-H1', '2014-H2', '2017-H1', '2019-H2', '2022-H1', '2024-H2']
fold=01 | train=27 val=9 test=10
  val : ['2004-H2', '2007-H1', '2009-H2', '2012-H1', '2014-H2', '2017-H1', '2019-H2', '2022-H1', '2024-H2']
  test: ['2002-H2', '2005-H1', '2007-H2', '2010-H1', '2012-H2', '2015-H1', '2017-H2', '2020-H1', '2022-H2', '2025-H1']
fold=02 | train=27 val=10 test=9
  val : ['2002-H2', '2005-H1', '2007-H2', '2010-H1', '2012-H2', '2015-H1', '2017-H2', '2020-H1', '2022-H2', '2025-H1']
  test: ['2003-H1', '2005-H2', '2008-H1', '2010-H2', '2013-H1', '2015-H2', '2018-H1', '2020-H2', '2023-H1']
fold=03 | train=28 val=9 test=9
  val : ['2003-H1', '2005-H2', '2008-H1', '2010-H2', '2013-H1', '2015-H2', '2018-H1', '2020-H2', '2023-H1']
  test: ['2003-H2', '2006-H1', '2008-H2', '2011-H1', '2013-H2', '2016-H1', '2018-H2', '2021-

## K-Fold (year_half) Debug

`legacy/2026` 방식과 동일하게 `year_half` 기준으로 fold를 회전시키며
train/val/test term 분할이 어떻게 되는지 확인합니다.

In [9]:
# helper 함수 기반 term 생성 확인 (_add_term_column)
term_df = _add_term_column(raw_df_with_time, time_col='Datetime', split_type=cfg.split_type)
terms = sorted(term_df['_term'].dropna().unique().tolist())

print('term count:', len(terms))
print('first 10 terms:', terms[:10])
print('last  10 terms:', terms[-10:])

term count: 46
first 10 terms: ['2002-H2', '2003-H1', '2003-H2', '2004-H1', '2004-H2', '2005-H1', '2005-H2', '2006-H1', '2006-H2', '2007-H1']
last  10 terms: ['2020-H2', '2021-H1', '2021-H2', '2022-H1', '2022-H2', '2023-H1', '2023-H2', '2024-H1', '2024-H2', '2025-H1']


In [10]:
# helper 함수 기반 fold 인덱스 계산 (_fold_indices)
for fold_numb in range(cfg.n_fold):
    fm = _fold_indices(len(terms), n_fold=cfg.n_fold, fold_numb=fold_numb)
    split_terms = {k: [terms[i] for i in v.tolist()] for k, v in fm.items()}
    print(f'\nfold={fold_numb}')
    print('  train:', split_terms['train'])
    print('  val  :', split_terms['val'])
    print('  test :', split_terms['test'])


fold=0
  train: ['2002-H2', '2003-H1', '2003-H2', '2005-H1', '2005-H2', '2006-H1', '2007-H2', '2008-H1', '2008-H2', '2010-H1', '2010-H2', '2011-H1', '2012-H2', '2013-H1', '2013-H2', '2015-H1', '2015-H2', '2016-H1', '2017-H2', '2018-H1', '2018-H2', '2020-H1', '2020-H2', '2021-H1', '2022-H2', '2023-H1', '2023-H2', '2025-H1']
  val  : ['2004-H1', '2006-H2', '2009-H1', '2011-H2', '2014-H1', '2016-H2', '2019-H1', '2021-H2', '2024-H1']
  test : ['2004-H2', '2007-H1', '2009-H2', '2012-H1', '2014-H2', '2017-H1', '2019-H2', '2022-H1', '2024-H2']

fold=1
  train: ['2003-H1', '2003-H2', '2004-H1', '2005-H2', '2006-H1', '2006-H2', '2008-H1', '2008-H2', '2009-H1', '2010-H2', '2011-H1', '2011-H2', '2013-H1', '2013-H2', '2014-H1', '2015-H2', '2016-H1', '2016-H2', '2018-H1', '2018-H2', '2019-H1', '2020-H2', '2021-H1', '2021-H2', '2023-H1', '2023-H2', '2024-H1']
  val  : ['2004-H2', '2007-H1', '2009-H2', '2012-H1', '2014-H2', '2017-H1', '2019-H2', '2022-H1', '2024-H2']
  test : ['2002-H2', '2005-H1', 

In [18]:
# 실제 DataModule.setup()을 개선된 k-fold 모드로 실행
cfg.fold_numb = 1
cfg_kfold = SimpleNamespace(**vars(cfg))

# Datetime 컬럼이 없는 CSV를 위해 임시 CSV 생성
# (실운영 데이터에 Datetime이 있으면 이 단계는 불필요)
tmp_df = raw_df.copy()
tmp_df['Datetime'] = pd.to_datetime(tmp_df[['Year', 'Month', 'Day', 'Hour']])
tmp_path = os.path.join('notebook', '_tmp_xray_with_datetime.csv')
tmp_df.to_csv(tmp_path, index=False)
cfg_kfold.data_path = str(tmp_path)

bundle_k = DataModule(cfg_kfold).setup()
print('k-fold input_size:', bundle_k.input_size)
print('k-fold target_index:', bundle_k.target_index)
print('k-fold train batches:', len(bundle_k.train_loader))
print('k-fold val batches  :', len(bundle_k.val_loader))
print('k-fold test batches :', len(bundle_k.test_loader))

bx, by = next(iter(bundle_k.train_loader))
print('k-fold batch_x shape:', tuple(bx.shape))
print('k-fold batch_y shape:', tuple(by.shape))

k-fold input_size: 5
k-fold target_index: 4
k-fold train batches: 3271
k-fold val batches  : 1079
k-fold test batches : 1063
k-fold batch_x shape: (32, 168, 5)
k-fold batch_y shape: (32, 72, 1)


In [20]:
# 실제 DataModule.setup()을 개선된 k-fold 모드로 실행
cfg.fold_numb = 3
cfg_kfold = SimpleNamespace(**vars(cfg))

# Datetime 컬럼이 없는 CSV를 위해 임시 CSV 생성
# (실운영 데이터에 Datetime이 있으면 이 단계는 불필요)
tmp_df = raw_df.copy()
tmp_df['Datetime'] = pd.to_datetime(tmp_df[['Year', 'Month', 'Day', 'Hour']])
tmp_path = os.path.join('notebook', '_tmp_xray_with_datetime.csv')
tmp_df.to_csv(tmp_path, index=False)
cfg_kfold.data_path = str(tmp_path)

bundle_k = DataModule(cfg_kfold).setup()
print('k-fold input_size:', bundle_k.input_size)
print('k-fold target_index:', bundle_k.target_index)
print('k-fold train batches:', len(bundle_k.train_loader))
print('k-fold val batches  :', len(bundle_k.val_loader))
print('k-fold test batches :', len(bundle_k.test_loader))

bx, by = next(iter(bundle_k.train_loader))
print('k-fold batch_x shape:', tuple(bx.shape))
print('k-fold batch_y shape:', tuple(by.shape))

k-fold input_size: 5
k-fold target_index: 4
k-fold train batches: 3235
k-fold val batches  : 1084
k-fold test batches : 1094
k-fold batch_x shape: (32, 168, 5)
k-fold batch_y shape: (32, 72, 1)


In [17]:
cfg.fold_numb

0

In [12]:
# 5) 개선된 k-fold 기준 최종 검증(assert)
assert bx.shape[1] == cfg_kfold.seq_len
assert by.shape[1] == cfg_kfold.pred_len
assert by.shape[-1] == 1
assert len(bundle_k.train_loader) > 0
assert len(bundle_k.val_loader) > 0
assert len(bundle_k.test_loader) > 0

print('k-fold assertions: PASS')

k-fold assertions: PASS


In [13]:
# (옵션) 문제 재현용: 짧은 시퀀스로 실패 케이스 확인
try:
    _ = _build_windows(values[:10], seq_len=24, pred_len=1)
except Exception as e:
    print('expected error:', type(e).__name__, str(e))

expected error: ValueError rows=10 is too short for seq_len+pred_len=25


In [14]:
# (선택) 임시 파일 정리
# tmp_path.unlink(missing_ok=True)